In [0]:
# ============================================================
# VALIDATION FINALE DE LA COUCHE BRONZE
# ============================================================

storage_account = "stsupplychain2026xx"

base_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "bronze/"
)

# ============================================================
# 1. LECTURE DES TABLES DELTA
# ============================================================

bronze_orders = (
    spark.read
    .format("delta")
    .load(base_path + "orders/")
)

bronze_shipments = (
    spark.read
    .format("delta")
    .load(base_path + "shipments/")
)

bronze_events = (
    spark.read
    .format("delta")
    .load(base_path + "delivery_events/")
)

bronze_carriers = (
    spark.read
    .format("delta")
    .load(base_path + "carriers/")
)

# ============================================================
# 2. COUNTS
# ============================================================

orders_count = bronze_orders.count()
shipments_count = bronze_shipments.count()
events_count = bronze_events.count()
carriers_count = bronze_carriers.count()

print("=== COUNTS BRONZE ===")

print("orders :", orders_count)
print("shipments :", shipments_count)
print("delivery_events :", events_count)
print("carriers :", carriers_count)

# ============================================================
# 3. RÉPARTITION PAR BATCH
# ============================================================

print("\n=== ORDERS PAR BATCH ===")

display(
    bronze_orders
    .groupBy("batch_id")
    .count()
    .orderBy("batch_id")
)

print("\n=== SHIPMENTS PAR BATCH ===")

display(
    bronze_shipments
    .groupBy("batch_id")
    .count()
    .orderBy("batch_id")
)

print("\n=== DELIVERY EVENTS PAR BATCH ===")

display(
    bronze_events
    .groupBy("batch_id")
    .count()
    .orderBy("batch_id")
)

print("\n=== CARRIERS PAR BATCH ===")

display(
    bronze_carriers
    .groupBy("batch_id")
    .count()
    .orderBy("batch_id")
)

# ============================================================
# 4. VÉRIFICATION DES MÉTADONNÉES
# ============================================================

required_metadata_columns = [
    "source_file",
    "batch_id",
    "ingestion_timestamp"
]

bronze_tables = {
    "orders": bronze_orders,
    "shipments": bronze_shipments,
    "delivery_events": bronze_events,
    "carriers": bronze_carriers
}

for table_name, dataframe in bronze_tables.items():

    missing_columns = [
        column
        for column in required_metadata_columns
        if column not in dataframe.columns
    ]

    assert len(missing_columns) == 0, (
        f"Colonnes de métadonnées manquantes dans "
        f"{table_name} : {missing_columns}"
    )

print(
    "\nToutes les colonnes de métadonnées "
    "sont présentes."
)

# ============================================================
# 5. APERÇU DES MÉTADONNÉES
# ============================================================

print("\n=== MÉTADONNÉES ORDERS ===")

display(
    bronze_orders
    .select(
        "order_id",
        "source_file",
        "batch_id",
        "ingestion_timestamp"
    )
    .orderBy("order_id")
    .limit(10)
)

# ============================================================
# 6. VALIDATIONS MINIMALES
# ============================================================

assert orders_count > 0, (
    "La table Bronze orders est vide."
)

assert shipments_count > 0, (
    "La table Bronze shipments est vide."
)

assert events_count > 0, (
    "La table Bronze delivery_events est vide."
)

assert carriers_count > 0, (
    "La table Bronze carriers est vide."
)

# ============================================================
# 7. RÉSULTAT FINAL
# ============================================================

print("\n======================================")
print("VALIDATION BRONZE RÉUSSIE")
print("======================================")

print(
    f"Orders          : {orders_count}"
)

print(
    f"Shipments       : {shipments_count}"
)

print(
    f"Delivery events : {events_count}"
)

print(
    f"Carriers        : {carriers_count}"
)

print(
    "\nLes quatre tables Bronze existent, "
    "contiennent des données et possèdent "
    "les métadonnées techniques attendues."
)